# 03_early_warning.ipynb

Objectives
- Validate **H3**: Can we flag high-risk sellers with enough lead time to act?
- Build a time-aware **(seller, date)** panel with:
  - Daily SLA + GMV metrics.
  - Rolling-window features (7 / 14 / 30 activity days).
  - Future severe-event labels at 7 / 14 / 21 calendar days.
- Train a simple, interpretable baseline model (logistic regression).
- Evaluate:
  - Global metrics: ROC AUC, Average Precision (PR-AUC).
  - Operational metrics: event recall@K, **GMV recall@K** for top-K flagged seller-days.

Notes
- Target = "at least one severe SLA violation in the next H days".
- GMV is used to quantify **GMV at risk captured** by the early-warning model.
- H4 (ROI simulation) will plug into the outputs of this notebook.

## 0. Import & Settings
This section:
- Configures the notebook environment.
- Registers the project root and `src` path.
- Imports:
  - Project-level utilities.
  - Early-warning feature builder (`features.early_warning`).
  - `scikit-learn` for modeling and evaluation.

In [3]:
import pandas as pd
import numpy as np
import altair as alt
from IPython.core.interactiveshell import InteractiveShell
from pathlib import Path
import sys

pd.set_option("display.max_columns", 50)
InteractiveShell.ast_node_interactivity = "all"
alt.data_transformers.disable_max_rows()

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Import project modules
from config import DATA_INTERIM, DATA_PROCESSED
from features.seller_metrics import validate_orders_sellers
# from features.early_warning import (
#     build_seller_daily_sla,
#     build_rolling_seller_features,
#     label_future_severe_events,
#     prepare_early_warning_panel,
# )
from data.preprocessing import load_orders_sellers

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Fixed RNG for reproducibility
RNG = np.random.RandomState(42)

DataTransformerRegistry.enable('default')

## 1. Build Daily Seller SLA + GMV Panel

- Loads `orders_sellers` (SSoT with SLA + GMV at order level).
- Validates its schema using `validate_orders_sellers`.
- Aggregates to a **daily seller-level** SLA + GMV panel using:
  - `build_seller_daily_sla`.
- Performs basic sanity checks on:
  - Tenure distribution.
  - Volume and GMV metrics.
  - Violation and severe rates.

In [6]:
# Load orders_sellers
orders_sellers = load_orders_sellers()

print(
    f"orders_sellers shape: {orders_sellers.shape}, "
    f"time range: {orders_sellers['order_purchase_timestamp'].min()} "
    f"to {orders_sellers['order_purchase_timestamp'].max()}"
)
orders_sellers.head()

orders_sellers shape: (99441, 13), time range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


,order_id,seller_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_sla_violation,is_severe_violation,has_time_anomaly,product_category_name_english,order_gmv
0,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,-8.0,False,False,False,housewares,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,289cdb325fb7e7f891c38608bf9e0962,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,-6.0,False,False,False,perfumery,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,4869f7a5dfa277a7dca6462dcf3b52b2,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,-18.0,False,False,False,auto,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,66922902710d126a0e7d26b0e3805106,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,-13.0,False,False,False,pet_shop,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,2c9e548be18521d1c43cde1c582c6de8,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,-10.0,False,False,False,stationery,28.62


In [7]:
# Validate structure and SLA field
validate_orders_sellers(orders_sellers, gmv_col="order_gmv")

orders_sellers contains unexpected order_status values: {'approved'}
